In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
# The input file has a 2-row header; pandas uses row 0 as column names,
# so row 1 (sub-headers) becomes data row 0 and is dropped later with iloc[1:]

df = pd.read_csv("../dataset/tcr_data_raw.tsv", sep='\t', low_memory=False)

df_filter = df[[
    "Epitope.1",    # epitope sequence
    "Chain 1",      # Chain 1 type (alpha / gamma)
    "Chain 1.1",    # Chain 1 host organism IRI
    "Assay.2",      # MHC allele names
    "Chain 1.3",    # TCR alpha Curated V gene
    "Chain 1.7",    # TCR alpha Curated J gene
    "Chain 1.11",   # TCR alpha CDR3 Curated
    "Chain 2.3",    # TCR beta Curated V gene
    "Chain 2.7",    # TCR beta Curated J gene
    "Chain 2.11",   # TCR beta CDR3 Curated
]]

df_cleaned = df_filter.dropna(subset=["Chain 1.11", "Chain 1.3", "Chain 2.3", "Chain 2.11"])

# Drop sub-header row (row 1 becomes data row 0 due to 2-row header format)
df_cleaned = df_cleaned.iloc[1:].reset_index(drop=True)

df_cleaned.columns = [
    "Epitope",
    "Chain1_Type", "Chain1_Organism", "MHC_Allele",
    "TCR_alpha_V", "TCR_alpha_J", "TCR_alpha_CDR3",
    "TCR_beta_V",  "TCR_beta_J",  "TCR_beta_CDR3",
]
print(f"After loading: {df_cleaned.shape}")
print(df_cleaned['Chain1_Type'].value_counts())

In [ ]:
# Filter 1: alpha chain only (excludes gamma-delta TCRs)
df_cleaned = df_cleaned[df_cleaned['Chain1_Type'] == 'alpha']

# Filter 2: human host only (NCBITaxon_9606)
df_cleaned = df_cleaned[df_cleaned['Chain1_Organism'].str.contains('9606', na=False)]

# Filter 3: MHC Class I only
# Keep entries with HLA allele but exclude HLA-D* (Class II: HLA-DR, HLA-DP, HLA-DQ)
df_cleaned = df_cleaned[
    df_cleaned['MHC_Allele'].str.contains('HLA', na=False) &
    ~df_cleaned['MHC_Allele'].str.contains('HLA-D', na=False)
]

df_cleaned = df_cleaned.reset_index(drop=True)
print(f"After species + chain type + MHC Class I filter: {df_cleaned.shape}")

In [ ]:
# Drop filter-only columns, reorder to match IEDB pipeline format
complete_pairs = df_cleaned[[
    'Epitope', 'TCR_alpha_V', 'TCR_alpha_CDR3',
    'TCR_beta_V', 'TCR_beta_CDR3', 'TCR_alpha_J', 'TCR_beta_J'
]].copy()

complete_pairs = complete_pairs.dropna().reset_index(drop=True)

# Keep only peptide epitopes (standard 20 amino acids) within length 8–15 aa
complete_pairs = complete_pairs[
    complete_pairs['Epitope'].str.match(r'^[ACDEFGHIKLMNPQRSTVWY]+$', na=False)
]
complete_pairs = complete_pairs[
    complete_pairs['Epitope'].str.len().between(8, 15)
].reset_index(drop=True)
print(f"After epitope filter (standard AA, 8–15 aa): {complete_pairs.shape}")

In [ ]:
# Remove stop codons ('*'), replace '#' with 'X', keep valid amino acids within length 8–20 aa
for col in ['TCR_alpha_CDR3', 'TCR_beta_CDR3']:
    complete_pairs = complete_pairs[~complete_pairs[col].str.contains(r'\*', na=False, regex=True)]
    complete_pairs[col] = complete_pairs[col].str.replace('#', 'X', regex=False)

valid_aa = r'^[ACDEFGHIKLMNPQRSTVWYX]+$'
complete_pairs = complete_pairs[
    complete_pairs['TCR_alpha_CDR3'].str.match(valid_aa, na=False) &
    complete_pairs['TCR_beta_CDR3'].str.match(valid_aa, na=False)
]

# CDR3 length filter: 8–20 aa
complete_pairs = complete_pairs[
    complete_pairs['TCR_alpha_CDR3'].str.len().between(8, 20) &
    complete_pairs['TCR_beta_CDR3'].str.len().between(8, 20)
].reset_index(drop=True)

print(f"After CDR3 cleaning + length filter (8–20 aa): {complete_pairs.shape}")
complete_pairs.head()

In [ ]:
complete_pairs.to_csv("../data/tcr_epitope_pairs.tsv", sep='\t', index=False)

In [ ]:
unique_tcrs = complete_pairs[['TCR_alpha_CDR3', 'TCR_beta_CDR3']].drop_duplicates().reset_index(drop=True)
print(f"Unique TCRs: {unique_tcrs.shape}")

unique_epitopes = complete_pairs[['Epitope']].drop_duplicates().reset_index(drop=True)
print(f"Unique epitopes: {unique_epitopes.shape}")

In [ ]:
unique_tcrs.to_csv("../data/unique_TCRs.tsv", sep='\t', index=False)
unique_epitopes.to_csv("../data/unique_epitopes.tsv", sep='\t', index=False)